# MLflow na prática — comparando modelos como no relatório WB Anywhere

Objetivo: pegar a lógica do PDF `r1 5 modelos.pdf` (5 modelos, várias métricas, réplicas, indicadores derivados) e reproduzir isso com **MLflow tracking**, que é exatamente a ferramenta que a equipe usa para comparar runs de experimentos.

Vamos construir em passos pequenos:

1. Imports e o que cada um faz
2. Conceitos do MLflow: experiment, run, params, metrics
3. Logar cada "modelo" do relatório como um run
4. Puxar tudo de volta como uma tabela (igual à Tabela 2 do PDF)
5. Calcular indicadores derivados (accuracy evidence-adjusted, clinical reliability score)
6. Comparar e ranquear

## 1. Imports

- `mlflow`: biblioteca principal — cria experiments, abre runs, loga params/metrics.
- `mlflow.set_tracking_uri`: diz onde o MLflow guarda os dados. Sem configurar nada, ele cria uma pasta `mlruns/` local — é o que vamos usar aqui.
- `pandas`: para transformar os resultados que voltam do MLflow em tabelas fáceis de comparar (igual às tabelas do PDF).

In [1]:
import mlflow
import pandas as pd

print("mlflow:", mlflow.__version__)
print("pandas:", pd.__version__)

mlflow: 3.15.2
pandas: 2.3.3


## 2. Conceitos rápidos

| Conceito | O que é | Equivalente no PDF |
|---|---|---|
| **Experiment** | Uma "pasta" que agrupa runs relacionados | O benchmark inteiro: "WB Anywhere — Benchmark R1" |
| **Run** | Uma execução específica, com seus params e métricas | Cada modelo avaliado (Gemini 3.7, GPT-5.2, Sonnet 5, ...) |
| **Param** | Configuração de entrada (não muda durante o run) | Nº de casos, nº de réplicas, campaign_id |
| **Metric** | Resultado medido (pode ter várias por run) | Accuracy, Grounding, Clinical safety, etc. |
| **Tag** | Metadado livre para filtrar/organizar | Ex: "familia: gemini", "modalidade: texto" |

Cada modelo do relatório vira **um run** dentro do **mesmo experiment**. Isso é o que permite comparar todos lado a lado depois.

In [ ]:
# Tracking local em SQLite: cria (ou reusa) um arquivo mlflow.db no diretório do notebook
# Obs: versões recentes do MLflow (3.x) bloqueiam por padrão o backend antigo baseado
# em pasta ("file:./mlruns") -- por isso usamos SQLite aqui.
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Cria (ou reusa, se já existir) o experiment
mlflow.set_experiment("wb_anywhere_benchmark_r1")

## 3. Os dados brutos (o que hoje está "preso" dentro do PDF)

No trabalho real, esses números viriam de rodar o benchmark de fato. Aqui vamos usar os valores consolidados da Tabela 2 do relatório (scores globais das 5 réplicas) só para praticar o fluxo de logging — o ponto não é decorar os números, é aprender o mecanismo.

In [4]:
# Um dicionário por modelo: params (config do run) + metrics (resultados)
modelos = {
    "gemini_3_7_flash": {
        "params": {"familia": "gemini", "casos": 194, "replicas": 5},
        "metrics": {
            "accuracy": 0.8158, "completeness": 0.8627, "relevance_v2": 0.8916,
            "grounding_v2": 0.5121, "clinical_safety": 0.9082, "tool_use": 0.7809,
            "citations": 0.9473, "image_fidelity": 0.8387,
        },
    },
    "gemini_3_6_flash": {
        "params": {"familia": "gemini", "casos": 194, "replicas": 5},
        "metrics": {
            "accuracy": 0.8153, "completeness": 0.8752, "relevance_v2": 0.9107,
            "grounding_v2": 0.4800, "clinical_safety": 0.9041, "tool_use": 0.7809,
            "citations": 0.9518, "image_fidelity": 0.8628,
        },
    },
    "gpt_5_4_mini": {
        "params": {"familia": "gpt", "casos": 194, "replicas": 5},
        "metrics": {
            "accuracy": 0.8068, "completeness": 0.7842, "relevance_v2": 0.8291,
            "grounding_v2": 0.7106, "clinical_safety": 0.9522, "tool_use": 0.7582,
            "citations": 0.9385, "image_fidelity": 0.9257,
        },
    },
    "gpt_5_2": {
        "params": {"familia": "gpt", "casos": 194, "replicas": 5},
        "metrics": {
            "accuracy": 0.8341, "completeness": 0.8419, "relevance_v2": 0.8792,
            "grounding_v2": 0.6771, "clinical_safety": 0.9415, "tool_use": 0.7485,
            "citations": 0.9968, "image_fidelity": 0.9161,
        },
    },
    "claude_sonnet_5": {
        "params": {"familia": "claude", "casos": 194, "replicas": 5},
        "metrics": {
            "accuracy": 0.7636, "completeness": 0.7749, "relevance_v2": 0.8283,
            "grounding_v2": 0.6474, "clinical_safety": 0.9105, "tool_use": 0.7819,
            "citations": 0.9615, "image_fidelity": 0.4415,
        },
    },
}

list(modelos.keys())

['gemini_3_7_flash',
 'gemini_3_6_flash',
 'gpt_5_4_mini',
 'gpt_5_2',
 'claude_sonnet_5']

## 4. Indicadores derivados

O relatório define dois indicadores calculados em cima das métricas oficiais:

- **Accuracy evidence-adjusted** = `0.60 * accuracy + 0.40 * disciplina_evidencia` (aqui vamos aproximar a disciplina de evidência pelo `grounding_v2`, só para termos algo calculável no exercício).
- **Clinical reliability score** = uma média ponderada que também usa grounding e segurança clínica.

Na vida real, essas fórmulas viriam de regras de negócio (gates, zero-hit, contradição etc). Aqui simplificamos para focar no mecanismo de logging.

In [5]:
def calcular_indicadores_derivados(m: dict) -> dict:
    accuracy_evidence_adjusted = 0.60 * m["accuracy"] + 0.40 * m["grounding_v2"]
    clinical_reliability_score = (
        0.35 * m["clinical_safety"]
        + 0.35 * m["grounding_v2"]
        + 0.30 * m["accuracy"]
    )
    return {
        "accuracy_evidence_adjusted": round(accuracy_evidence_adjusted, 4),
        "clinical_reliability_score": round(clinical_reliability_score, 4),
    }

# teste rápido com um modelo
calcular_indicadores_derivados(modelos["gpt_5_2"]["metrics"])

{'accuracy_evidence_adjusted': 0.7713, 'clinical_reliability_score': 0.8167}

## 5. Logando cada modelo como um run do MLflow

Padrão típico:

```python
with mlflow.start_run(run_name="..."):
    mlflow.log_params({...})
    mlflow.log_metrics({...})
```

Tudo dentro do `with` pertence ao mesmo run; ao sair do bloco, o MLflow fecha o run automaticamente.

In [ ]:
for nome_modelo, dados in modelos.items():
    with mlflow.start_run(run_name=nome_modelo):
        mlflow.log_params(dados["params"])
        mlflow.log_metrics(dados["metrics"])

        derivados = calcular_indicadores_derivados(dados["metrics"])
        mlflow.log_metrics(derivados)

        mlflow.set_tag("benchmark", "wb_anywhere_r1")

print("5 runs logados.")

## 6. Puxando tudo de volta como tabela comparativa

`mlflow.search_runs()` devolve um DataFrame do pandas com uma linha por run e uma coluna por param/metric — é assim que se monta a "Tabela 2" do PDF programaticamente, em vez de copiar números à mão.

In [ ]:
experiment = mlflow.get_experiment_by_name("wb_anywhere_benchmark_r1")
df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# As colunas de métrica vêm prefixadas com "metrics." e params com "params."
colunas_interessantes = [c for c in df.columns if c.startswith("metrics.") or c == "tags.mlflow.runName"]
tabela = df[colunas_interessantes].rename(columns=lambda c: c.replace("metrics.", ""))
tabela = tabela.rename(columns={"tags.mlflow.runName": "modelo"}).set_index("modelo")
tabela.sort_values("clinical_reliability_score", ascending=False)

## 7. Ranking, do jeito que o relatório conclui

O PDF conclui que o GPT-5.2 lidera os dois indicadores derivados. Vamos confirmar isso a partir da nossa tabela, em vez de confiar de olho.

In [ ]:
ranking = tabela[["accuracy_evidence_adjusted", "clinical_reliability_score"]].sort_values(
    "clinical_reliability_score", ascending=False
)
print("Líder em clinical_reliability_score:", ranking.index[0])
ranking

## 8. Ver isso visualmente: a MLflow UI

No terminal, dentro desta pasta (`03-mlflow/`), rode:

```bash
mlflow ui
```

e abra `http://127.0.0.1:5000`. Lá dá pra ver o experiment `wb_anywhere_benchmark_r1`, comparar os 5 runs lado a lado, ordenar por métrica e até plotar gráficos — é a mesma ideia da Tabela 2 do PDF, só que interativa.

## Próximos passos sugeridos

1. Troque os números fixos por uma função que realmente chama os modelos (LangChain) e calcula as métricas de verdade.
2. Adicione múltiplas réplicas por modelo (5 runs por modelo, com uma tag `replica=1..5`) e depois agregue com `groupby` no pandas — é assim que o relatório calcula médias e variabilidade entre réplicas.
3. Use `mlflow.log_artifact()` para salvar o próprio PDF/relatório final gerado como artefato do experiment.